In [0]:
%pip install pyinstaller

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
pip install PyPDF2

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# import tkinter as tk
# from tkinter import filedialog, messagebox
# from PyPDF2 import PdfMerger

# class PDFConcatenatorApp:
#     def __init__(self, root):
#         self.root = root
#         self.root.title("Concatenator de fisiere PDF")
#         self.root.geometry("500x400")

#         self.pdf_files = []

#         self.label = tk.Label(root, text="Fisiere selectate:", anchor="w")
#         self.label.pack(fill="x", padx=10, pady=(10, 0))

#         self.file_listbox = tk.Listbox(root, selectmode=tk.SINGLE)
#         self.file_listbox.pack(fill="both", expand=True, padx=10, pady=5)

#         self.add_button = tk.Button(root, text="adauga fisiere PDF", command=self.add_files)
#         self.add_button.pack(padx=10, pady=5)

#         self.remove_button = tk.Button(root, text="sterge un fisier selectat pentru concatenare", command=self.remove_file)
#         self.remove_button.pack(padx=10, pady=5)

#         self.concat_button = tk.Button(root, text="concatenare PDF-uri", command=self.concatenate_pdfs)
#         self.concat_button.pack(padx=10, pady=5)
#         self.exit_button = tk.Button(root, text="Exit", command=self.root.destroy)
#         #self.exit_button = tk.Button(root, text="Exit", command=self.root.quit)
#         self.exit_button.pack(padx=10, pady=5)

#     def add_files(self):
#         files = filedialog.askopenfilenames(filetypes=[("PDF files", "*.pdf")])
#         for file in files:
#             if file not in self.pdf_files:
#                 self.pdf_files.append(file)
#                 self.file_listbox.insert(tk.END, file)

#     def remove_file(self):
#         selected = self.file_listbox.curselection()
#         if selected:
#             index = selected[0]
#             self.file_listbox.delete(index)
#             del self.pdf_files[index]

#     def concatenate_pdfs(self):
#         if len(self.pdf_files) < 2:
#             messagebox.showwarning("Atenție", "alege minim 2 fisiere PDF.")
#             return

#         output_file = filedialog.asksaveasfilename(defaultextension=".pdf", filetypes=[("PDF files", "*.pdf")])
#         if not output_file:
#             return

#         try:
#             merger = PdfMerger()
#             for pdf in self.pdf_files:
#                 merger.append(pdf)
#             merger.write(output_file)
#             merger.close()
#             messagebox.showinfo("Gata...", f"Documentul a fost salvat: {output_file}")
#         except Exception as e:
#             messagebox.showerror("Eroare", f"eroare la salvare: {str(e)}")

# if __name__ == "__main__":
#     root = tk.Tk()
#     app = PDFConcatenatorApp(root)
#     root.mainloop()

In [0]:
%pip install ipywidgets PyPDF2

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import io
import base64
from PyPDF2 import PdfMerger
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

class PDFConcatenatorApp:
    def __init__(self):
        self.pdf_files = []

        self.title = HTML("<h3>Concatenator de fisiere PDF</h3>")

        self.file_listbox = widgets.Select(
            options=[],
            rows=10,
            description="Fisiere:"
        )

        self.upload = widgets.FileUpload(accept=".pdf", multiple=True)
        self.add_button = widgets.Button(description="Adauga fisiere PDF", button_style="primary")
        self.remove_button = widgets.Button(description="Sterge fisier selectat")
        self.concat_button = widgets.Button(description="Concatenare PDF-uri", button_style="success")
        self.exit_button = widgets.Button(description="Exit", button_style="danger")

        self.status = widgets.Output()

        self.controls = widgets.VBox([
            self.upload,
            widgets.HBox([self.add_button, self.remove_button]),
            self.file_listbox,
            self.concat_button,
            self.exit_button,
            self.status
        ])

        self.add_button.on_click(self.add_files)
        self.remove_button.on_click(self.remove_file)
        self.concat_button.on_click(self.concatenate_pdfs)
        self.exit_button.on_click(self.exit_app)

        display(self.title, self.controls)

    def refresh_listbox(self):
        self.file_listbox.options = [f["name"] for f in self.pdf_files]

    def add_files(self, _):
        added = 0
        for key, meta in self.upload.value.items():
            name = meta.get("metadata", {}).get("name") or meta.get("name") or key
            data = meta.get("content") or meta.get("data")
            if name and data and not any(f["name"] == name for f in self.pdf_files):
                self.pdf_files.append({"name": name, "data": data})
                added += 1

        with self.status:
            clear_output()
            if added:
                print(f"Adaugat(e) {added} fisier(e).")
            else:
                print("Niciun fisier nou adaugat.")

        self.refresh_listbox()
        self.upload.value.clear()
        self.upload._counter = 0

    def remove_file(self, _):
        idx = self.file_listbox.index
        if idx is None or idx < 0 or idx >= len(self.pdf_files):
            with self.status:
                clear_output()
                print("Selecteaza un fisier de sters.")
            return
        removed = self.pdf_files.pop(idx)
        self.refresh_listbox()
        with self.status:
            clear_output()
            print(f"Fisier sters: {removed['name']}")

    def concatenate_pdfs(self, _):
        if len(self.pdf_files) < 2:
            with self.status:
                clear_output()
                print("Alege cel putin doua fisiere PDF pentru concatenare.")
            return

        try:
            merger = PdfMerger()
            for f in self.pdf_files:
                merger.append(io.BytesIO(f["data"]))

            out_path = "/tmp/concatenat.pdf"
            with open(out_path, "wb") as f_out:
                merger.write(f_out)
            merger.close()

            with open(out_path, "rb") as f_out:
                b64 = base64.b64encode(f_out.read()).decode("utf-8")
            link = f'<a download="concatenat.pdf" href="data:application/pdf;base64,{b64}">Descarca fisierul concatenat</a>'

            with self.status:
                clear_output()
                print("Concatenare reusita.")
                display(HTML(link))
        except Exception as e:
            with self.status:
                clear_output()
                print(f"Eroare la concatenare: {e}")

    def exit_app(self, _):
        with self.status:
            clear_output()
            print("Aplicatia a fost inchisa.")
        self.controls.close()
        self.title.close()

app = PDFConcatenatorApp()